In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
print(os.listdir('/content/drive/MyDrive'))

['module1', 'member1_parsing.ipynb']


In [3]:
import os

base = '/content/drive/MyDrive/module1'

folders = [
    f'{base}/data/raw',
    f'{base}/data/processed',
    f'{base}/code',
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f'✓ {folder}')

✓ /content/drive/MyDrive/module1/data/raw
✓ /content/drive/MyDrive/module1/data/processed
✓ /content/drive/MyDrive/module1/code


In [4]:
!pip install drain3 tqdm pandas numpy --quiet

  Preparing metadata (setup.py) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyiceberg 0.12.0 requires cachetools>=5.5, but you have cachetools 4.2.1 which is incompatible.


In [6]:
from drain3 import TemplateMiner
from drain3.template_miner_config import TemplateMinerConfig

config = TemplateMinerConfig()
config.drain_sim_th = 0.4
config.drain_depth = 4

miner = TemplateMiner(config=config)

test_line = "081109 203615 148 INFO dfs.DataNode$PacketResponder: PacketResponder 1 for block blk_38865049064139660 terminating"
result = miner.add_log_message(test_line)

print("Template mined:", result["template_mined"])
print("Cluster ID:", result["cluster_id"])

Template mined: 081109 203615 148 INFO dfs.DataNode$PacketResponder: PacketResponder 1 for block blk_38865049064139660 terminating
Cluster ID: 1


In [7]:
import os

base = '/content/drive/MyDrive/module1'

print('Folder structure:')
for root, dirs, files in os.walk(base):
    level = root.replace(base, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')

test_file = f'{base}/data/processed/_test.txt'
with open(test_file, 'w') as f:
    f.write('Member 1 workspace is ready.')

with open(test_file, 'r') as f:
    print(f'\nTest file content: {f.read()}')

os.remove(test_file)
print('\n✓ Step 1 complete — workspace ready.')

Folder structure:
module1/
  data/
    raw/
    processed/
  code/

Test file content: Member 1 workspace is ready.

✓ Step 1 complete — workspace ready.


In [8]:
import os

shortcut_path = '/content/drive/MyDrive/LogSentry_Capstone'

print('Shortcut exists?', os.path.exists(shortcut_path))

if os.path.exists(shortcut_path):
    print('\nTop-level contents:')
    for item in os.listdir(shortcut_path):
        print(f'  {item}')

Shortcut exists? True

Top-level contents:
  01_Dataset
  02_Parsed_Data
  03_Models
  04_Results


In [9]:
import os

zip_path = '/content/drive/MyDrive/LogSentry_Capstone/01_Dataset/HDFS_v1.zip'

print('Zip exists?', os.path.exists(zip_path))

if os.path.exists(zip_path):
    size_gb = os.path.getsize(zip_path) / (1024**3)
    print(f'Size: {size_gb:.2f} GB')

Zip exists? True
Size: 0.17 GB


In [10]:
import zipfile

zip_path = '/content/drive/MyDrive/LogSentry_Capstone/01_Dataset/HDFS_v1.zip'

try:
    with zipfile.ZipFile(zip_path, 'r') as z:
        bad_file = z.testzip()  # returns None if OK, else name of first bad file
        if bad_file is None:
            print('✅ Zip is valid (no CRC errors).')
        else:
            print(f'❌ Corrupt entry: {bad_file}')

        print('\nContents of the zip:')
        for info in z.infolist():
            size_mb = info.file_size / (1024**2)
            comp_mb = info.compress_size / (1024**2)
            print(f'  {info.filename}  ({size_mb:.1f} MB uncompressed → {comp_mb:.1f} MB compressed)')
except zipfile.BadZipFile as e:
    print(f'❌ Not a valid zip file: {e}')

✅ Zip is valid (no CRC errors).

Contents of the zip:
  HDFS.log  (1504.9 MB uncompressed → 141.8 MB compressed)
  preprocessed/  (0.0 MB uncompressed → 0.0 MB compressed)
  preprocessed/anomaly_label.csv  (17.8 MB uncompressed → 5.9 MB compressed)
  preprocessed/Event_occurrence_matrix.csv  (50.6 MB uncompressed → 6.7 MB compressed)
  preprocessed/Event_traces.csv  (119.6 MB uncompressed → 16.7 MB compressed)
  preprocessed/HDFS.log_templates.csv  (0.0 MB uncompressed → 0.0 MB compressed)
  preprocessed/HDFS.npz  (50.5 MB uncompressed → 6.9 MB compressed)
  README.md  (0.0 MB uncompressed → 0.0 MB compressed)


In [11]:
import shutil
import os
from tqdm import tqdm

src = '/content/drive/MyDrive/LogSentry_Capstone/01_Dataset/HDFS_v1.zip'
dst = '/content/drive/MyDrive/module1/data/raw/HDFS_v1.zip'

print('Copying zip into your module1/...')
shutil.copy2(src, dst)

size_mb = os.path.getsize(dst) / (1024**2)
print(f'✅ Copied. Size: {size_mb:.1f} MB')

Copying zip into your module1/...
✅ Copied. Size: 178.0 MB


In [12]:
import zipfile
import os

zip_path = '/content/drive/MyDrive/module1/data/raw/HDFS_v1.zip'
extract_to = '/content/drive/MyDrive/module1/data/raw/'

with zipfile.ZipFile(zip_path, 'r') as z:
    member = 'preprocessed/anomaly_label.csv'

    with z.open(member) as src, open(extract_to + 'anomaly_label.csv', 'wb') as dst:
        while True:
            chunk = src.read(1024 * 1024)
            if not chunk:
                break
            dst.write(chunk)

size_mb = os.path.getsize(extract_to + 'anomaly_label.csv') / (1024**2)
print(f'✅ anomaly_label.csv extracted. Size: {size_mb:.1f} MB')

✅ anomaly_label.csv extracted. Size: 17.8 MB


In [13]:
import os

raw_dir = '/content/drive/MyDrive/module1/data/raw/'

print('Contents of module1/data/raw/:')
for f in sorted(os.listdir(raw_dir)):
    path = os.path.join(raw_dir, f)
    size_mb = os.path.getsize(path) / (1024**2)
    print(f'  {f}  ({size_mb:.1f} MB)')

Contents of module1/data/raw/:
  HDFS_v1.zip  (178.0 MB)
  anomaly_label.csv  (17.8 MB)


In [14]:
import zipfile
import time

zip_path = '/content/drive/MyDrive/module1/data/raw/HDFS_v1.zip'
sample_path = '/content/sample_hdfs.log'

N_LINES = 10000

start = time.time()
with zipfile.ZipFile(zip_path, 'r') as z:
    with z.open('HDFS.log') as f_in, open(sample_path, 'w', encoding='utf-8', errors='ignore') as f_out:
        for i, line in enumerate(f_in):
            if i >= N_LINES:
                break
            f_out.write(line.decode('utf-8', errors='ignore'))

elapsed = time.time() - start
print(f'✅ Extracted {N_LINES} lines in {elapsed:.1f}s')
print(f'Saved to: {sample_path}')

✅ Extracted 10000 lines in 0.0s
Saved to: /content/sample_hdfs.log


In [15]:
with open('/content/sample_hdfs.log', 'r') as f:
    for i in range(10):
        print(f.readline().strip())

081109 203518 143 INFO dfs.DataNode$DataXceiver: Receiving block blk_-1608999687919862906 src: /10.250.19.102:54106 dest: /10.250.19.102:50010
081109 203518 35 INFO dfs.FSNamesystem: BLOCK* NameSystem.allocateBlock: /mnt/hadoop/mapred/system/job_200811092030_0001/job.jar. blk_-1608999687919862906
081109 203519 143 INFO dfs.DataNode$DataXceiver: Receiving block blk_-1608999687919862906 src: /10.250.10.6:40524 dest: /10.250.10.6:50010
081109 203519 145 INFO dfs.DataNode$DataXceiver: Receiving block blk_-1608999687919862906 src: /10.250.14.224:42420 dest: /10.250.14.224:50010
081109 203519 145 INFO dfs.DataNode$PacketResponder: PacketResponder 1 for block blk_-1608999687919862906 terminating
081109 203519 145 INFO dfs.DataNode$PacketResponder: PacketResponder 2 for block blk_-1608999687919862906 terminating
081109 203519 145 INFO dfs.DataNode$PacketResponder: Received block blk_-1608999687919862906 of size 91178 from /10.250.10.6
081109 203519 145 INFO dfs.DataNode$PacketResponder: Receiv

In [16]:
from drain3 import TemplateMiner
from drain3.template_miner_config import TemplateMinerConfig
from tqdm import tqdm
import time

config = TemplateMinerConfig()
config.drain_sim_th = 0.4
config.drain_depth = 4
config.drain_max_children = 100
config.profiling_enabled = False

miner = TemplateMiner(config=config)

start = time.time()
with open('/content/sample_hdfs.log', 'r', encoding='utf-8', errors='ignore') as f:
    for line in tqdm(f, total=10000, desc='Parsing'):
        line = line.strip()
        if line:
            miner.add_log_message(line)
elapsed = time.time() - start

print(f'\n✅ Parsed 10,000 lines in {elapsed:.1f}s')
print(f'Discovered {len(miner.drain.clusters)} unique templates')

Parsing: 100%|██████████| 10000/10000 [00:00<00:00, 57508.45it/s]


✅ Parsed 10,000 lines in 0.2s
Discovered 16 unique templates


In [17]:
clusters = sorted(miner.drain.clusters, key=lambda c: c.size, reverse=True)

print(f'{"ID":<6} {"Size":<8} {"Template"}')
print('-' * 100)
for c in clusters[:20]:
    template = c.get_template()
    print(f'{c.cluster_id:<6} {c.size:<8} {template[:90]}')

ID     Size     Template
----------------------------------------------------------------------------------------------------
1      3031     081109 <*> <*> INFO dfs.DataNode$DataXceiver: Receiving block <*> src: <*> dest: <*>
3      1851     081109 <*> <*> INFO dfs.DataNode$PacketResponder: PacketResponder <*> for block <*> termin
4      1850     081109 <*> <*> INFO dfs.DataNode$PacketResponder: Received block <*> of size <*> from <*>
5      1827     081109 <*> <*> INFO dfs.FSNamesystem: BLOCK* NameSystem.addStoredBlock: blockMap updated: 
2      1012     081109 <*> <*> INFO dfs.FSNamesystem: BLOCK* NameSystem.allocateBlock: <*> <*>
9      407      081109 <*> <*> INFO dfs.DataNode$DataXceiver: <*> Served block <*> to <*>
6      7        081109 <*> <*> INFO dfs.DataNode$DataXceiver: Received block blk_-1608999687919862906 src:
7      4        081109 <*> <*> INFO dfs.DataNode$DataTransfer: <*> block blk_-1608999687919862906 to <*>
8      2        081109 203521 19 INFO <*> <*> <*> <*> to

In [18]:
import zipfile
import pandas as pd
from io import BytesIO

zip_path = '/content/drive/MyDrive/module1/data/raw/HDFS_v1.zip'

with zipfile.ZipFile(zip_path, 'r') as z:
    with z.open('preprocessed/HDFS.log_templates.csv') as f:
        paper_templates = pd.read_csv(BytesIO(f.read()))

print(f"Paper's parser found {len(paper_templates)} templates")
print('\nSample:')
print(paper_templates.head(10).to_string())

Paper's parser found 29 templates

Sample:
  EventId                                    EventTemplate
0      E1           [*]Adding an already existing block[*]
1      E2                 [*]Verification succeeded for[*]
2      E3                          [*]Served block[*]to[*]
3      E4           [*]Got exception while serving[*]to[*]
4      E5             [*]Receiving block[*]src:[*]dest:[*]
5      E6    [*]Received block[*]src:[*]dest:[*]of size[*]
6      E7            [*]writeBlock[*]received exception[*]
7      E8  [*]PacketResponder[*]for block[*]Interrupted[*]
8      E9            [*]Received block[*]of size[*]from[*]
9     E10                [*]PacketResponder[*]Exception[*]


In [19]:
import zipfile
import os
import time

ZIP_PATH = '/content/drive/MyDrive/module1/data/raw/HDFS_v1.zip'
LOCAL_LOG = '/content/HDFS.log'

if os.path.exists(LOCAL_LOG):
    print(f'{LOCAL_LOG} already exists, skipping extraction.')
else:
    print('Extracting HDFS.log to Colab local disk...')
    start = time.time()
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extract('HDFS.log', path='/content/')
    elapsed = time.time() - start
    size_gb = os.path.getsize(LOCAL_LOG) / (1024**3)
    print(f'✅ Extracted in {elapsed:.1f}s. Size: {size_gb:.2f} GB')

Extracting HDFS.log to Colab local disk...
✅ Extracted in 15.5s. Size: 1.47 GB


In [20]:
import re
import time
from drain3 import TemplateMiner
from drain3.template_miner_config import TemplateMinerConfig
from tqdm import tqdm

LOCAL_LOG = '/content/HDFS.log'
LOCAL_PARSED = '/content/parsed_logs.csv'

# Extract block_id from raw log line
BLOCK_RE = re.compile(r'(blk_-?\d+)')

# --- Configure Drain (same as Step 3) ---
config = TemplateMinerConfig()
config.drain_sim_th = 0.4
config.drain_depth = 4
config.drain_max_children = 100
config.profiling_enabled = False

miner = TemplateMiner(config=config)
template_map = {}  # template_text -> log_key (int)

# --- Count total lines for progress bar ---
print('Counting lines for progress bar...')
with open(LOCAL_LOG, 'rb') as f:
    total_lines = sum(1 for _ in f)
print(f'Total lines: {total_lines:,}')

# --- Parse ---
start = time.time()
parsed = 0
no_block = 0

with open(LOCAL_LOG, 'r', encoding='utf-8', errors='ignore') as f_in, \
     open(LOCAL_PARSED, 'w', encoding='utf-8') as f_out:

    f_out.write('line_number,log_key,block_id\n')

    for i, line in enumerate(tqdm(f_in, total=total_lines, desc='Parsing')):
        line = line.strip()
        if not line:
            continue

        result = miner.add_log_message(line)
        template_text = result['template_mined']

        if template_text not in template_map:
            template_map[template_text] = len(template_map) + 1
        log_key = template_map[template_text]

        m = BLOCK_RE.search(line)
        block_id = m.group(1) if m else ''
        if not block_id:
            no_block += 1

        f_out.write(f'{i},{log_key},{block_id}\n')
        parsed += 1

elapsed = time.time() - start
print(f'\n✅ Parsed {parsed:,} lines in {elapsed/60:.1f} minutes')
print(f'Unique templates: {len(template_map)}')
print(f'Lines without block_id: {no_block:,} ({100*no_block/max(parsed,1):.2f}%)')

Counting lines for progress bar...
Total lines: 11,175,629


Parsing: 100%|██████████| 11175629/11175629 [04:59<00:00, 37263.58it/s]


✅ Parsed 11,175,629 lines in 5.0 minutes
Unique templates: 157
Lines without block_id: 0 (0.00%)


In [21]:
import re

# Regex masks (adapted from LogPai / DeepLog paper)
# Order matters: run most specific first
MASK_PATTERNS = [
    (re.compile(r'blk_-?\d+'), 'BLK'),               # block IDs
    (re.compile(r'/(\d{1,3}\.){3}\d{1,3}(:\d+)?'), 'IP'),  # IP:port or /IP
    (re.compile(r'\b\d{1,3}(\.\d{1,3}){3}\b'), 'IP'),      # bare IP
    (re.compile(r'(?<=[^A-Za-z0-9])\d+(?=[^A-Za-z0-9])'), 'NUM'),  # standalone numbers
]

def mask_line(line):
    for pattern, replacement in MASK_PATTERNS:
        line = pattern.sub(replacement, line)
    return line

# Sanity test
test = "081109 203518 143 INFO dfs.DataNode$DataXceiver: Receiving block blk_-1608999687919862906 src: /10.250.19.102:54106 dest: /10.250.19.102:50010"
print("Before:", test)
print("After: ", mask_line(test))

Before: 081109 203518 143 INFO dfs.DataNode$DataXceiver: Receiving block blk_-1608999687919862906 src: /10.250.19.102:54106 dest: /10.250.19.102:50010
After:  081109 NUM NUM INFO dfs.DataNode$DataXceiver: Receiving block BLK src: IP dest: IP


In [22]:
import time
from drain3 import TemplateMiner
from drain3.template_miner_config import TemplateMinerConfig
from tqdm import tqdm

LOCAL_LOG = '/content/HDFS.log'
LOCAL_PARSED = '/content/parsed_logs.csv'          # overwrites old one
LOCAL_TEMPLATES = '/content/templates.csv'

BLOCK_RE = re.compile(r'(blk_-?\d+)')  # extract from ORIGINAL line, not masked

config = TemplateMinerConfig()
config.drain_sim_th = 0.4
config.drain_depth = 4
config.drain_max_children = 100
config.profiling_enabled = False

miner = TemplateMiner(config=config)
template_map = {}

with open(LOCAL_LOG, 'rb') as f:
    total_lines = sum(1 for _ in f)

start = time.time()
parsed = 0
no_block = 0

with open(LOCAL_LOG, 'r', encoding='utf-8', errors='ignore') as f_in, \
     open(LOCAL_PARSED, 'w', encoding='utf-8') as f_out:

    f_out.write('line_number,log_key,block_id\n')

    for i, line in enumerate(tqdm(f_in, total=total_lines, desc='Parsing (masked)')):
        line = line.strip()
        if not line:
            continue

        # Extract block_id from ORIGINAL line (before masking)
        m = BLOCK_RE.search(line)
        block_id = m.group(1) if m else ''
        if not block_id:
            no_block += 1

        # Mask then feed to Drain
        masked = mask_line(line)
        result = miner.add_log_message(masked)
        template_text = result['template_mined']

        if template_text not in template_map:
            template_map[template_text] = len(template_map) + 1
        log_key = template_map[template_text]

        f_out.write(f'{i},{log_key},{block_id}\n')
        parsed += 1

elapsed = time.time() - start
print(f'\n✅ Parsed {parsed:,} lines in {elapsed/60:.1f} minutes')
print(f'Unique templates: {len(template_map)}')
print(f'Lines without block_id: {no_block:,} ({100*no_block/max(parsed,1):.4f}%)')

Parsing (masked): 100%|██████████| 11175629/11175629 [07:57<00:00, 23402.20it/s]


✅ Parsed 11,175,629 lines in 8.0 minutes
Unique templates: 96
Lines without block_id: 0 (0.0000%)


In [23]:
# 4B.3 — Save templates
with open('/content/templates.csv', 'w', encoding='utf-8') as f:
    f.write('log_key,template\n')
    for template, key in template_map.items():
        t = template.replace('"', '""')
        f.write(f'{key},"{t}"\n')
print(f'✅ Templates saved: {len(template_map)}')

✅ Templates saved: 96


In [24]:
# 4B.4 — Inspect top templates AND the long tail
clusters = sorted(miner.drain.clusters, key=lambda c: c.size, reverse=True)

print('=== TOP 25 TEMPLATES (by frequency) ===')
print(f'{"ID":<6} {"Size":<10} {"Template"}')
print('-' * 110)
for c in clusters[:25]:
    print(f'{c.cluster_id:<6} {c.size:<10} {c.get_template()[:100]}')

print('\n=== SIZE DISTRIBUTION ===')
sizes = [c.size for c in clusters]
import numpy as np
print(f'Total clusters: {len(sizes)}')
print(f'Max size: {max(sizes):,}')
print(f'Clusters with size == 1: {sum(1 for s in sizes if s == 1)}')
print(f'Clusters with size <= 5: {sum(1 for s in sizes if s <= 5)}')
print(f'Clusters with size <= 100: {sum(1 for s in sizes if s <= 100)}')
print(f'Clusters covering top 99% of lines: need to check cumulative')
cumsum = np.cumsum(sorted(sizes, reverse=True))
total = sum(sizes)
n_99 = int(np.searchsorted(cumsum, 0.99 * total) + 1)
print(f'→ {n_99} clusters cover 99% of all log lines')

=== TOP 25 TEMPLATES (by frequency) ===
ID     Size       Template
--------------------------------------------------------------------------------------------------------------
1      1723232    <*> NUM NUM INFO dfs.DataNode$DataXceiver: Receiving block BLK src: IP dest: IP
5      1719741    <*> NUM NUM INFO dfs.FSNamesystem: BLOCK* NameSystem.addStoredBlock: blockMap updated: IP:NUM is add
3      1706728    <*> NUM NUM INFO dfs.DataNode$PacketResponder: PacketResponder NUM for block BLK <*>
4      1706514    <*> NUM NUM INFO dfs.DataNode$PacketResponder: Received block BLK of size NUM from IP
24     1402854    <*> NUM NUM INFO dfs.FSNamesystem: BLOCK* <*> <*> <*> <*> <*> <*> <*> IP:50010
37     764631     081111 NUM NUM INFO <*> <*> <*> <*> <*> <*>
2      695097     <*> NUM NUM INFO <*> <*> <*> <*> BLK
29     640152     081110 NUM NUM INFO <*> <*> <*> <*> <*> <*>
9      428726     <*> NUM NUM INFO dfs.DataNode$DataXceiver: IP:NUM Served block BLK to IP
27     356207     <*> NUM NUM W

In [25]:
import re

MASK_PATTERNS = [
    (re.compile(r'^\d{6}\s+'), 'DATE '),                   # date at line start
    (re.compile(r'blk_-?\d+'), 'BLK'),                     # block IDs
    (re.compile(r'/(\d{1,3}\.){3}\d{1,3}(:\d+)?'), 'IP'),  # /IP:port
    (re.compile(r'\b\d{1,3}(\.\d{1,3}){3}\b'), 'IP'),      # bare IP
    (re.compile(r'(?<=[^A-Za-z0-9])\d+(?=[^A-Za-z0-9])'), 'NUM'),
]

def mask_line(line):
    for pattern, replacement in MASK_PATTERNS:
        line = pattern.sub(replacement, line)
    return line

# Sanity check
test = "081109 203518 143 INFO dfs.DataNode$DataXceiver: Receiving block blk_-1608999687919862906 src: /10.250.19.102:54106 dest: /10.250.19.102:50010"
print("Before:", test)
print("After: ", mask_line(test))

Before: 081109 203518 143 INFO dfs.DataNode$DataXceiver: Receiving block blk_-1608999687919862906 src: /10.250.19.102:54106 dest: /10.250.19.102:50010
After:  DATE NUM NUM INFO dfs.DataNode$DataXceiver: Receiving block BLK src: IP dest: IP


In [26]:
import time, re, os
from drain3 import TemplateMiner
from drain3.template_miner_config import TemplateMinerConfig
from tqdm import tqdm

LOCAL_LOG = '/content/HDFS.log'
LOCAL_PARSED = '/content/parsed_logs.csv'
LOCAL_TEMPLATES = '/content/templates.csv'

BLOCK_RE = re.compile(r'(blk_-?\d+)')

config = TemplateMinerConfig()
config.drain_sim_th = 0.4
config.drain_depth = 4
config.drain_max_children = 100
config.profiling_enabled = False

miner = TemplateMiner(config=config)

with open(LOCAL_LOG, 'rb') as f:
    total_lines = sum(1 for _ in f)

start = time.time()
parsed = 0
no_block = 0

with open(LOCAL_LOG, 'r', encoding='utf-8', errors='ignore') as f_in, \
     open(LOCAL_PARSED, 'w', encoding='utf-8') as f_out:

    f_out.write('line_number,log_key,block_id\n')

    for i, line in enumerate(tqdm(f_in, total=total_lines, desc='Final parse')):
        line = line.strip()
        if not line:
            continue

        m = BLOCK_RE.search(line)
        block_id = m.group(1) if m else ''
        if not block_id:
            no_block += 1

        masked = mask_line(line)
        result = miner.add_log_message(masked)
        log_key = result['cluster_id']          # ← THE FIX

        f_out.write(f'{i},{log_key},{block_id}\n')
        parsed += 1

elapsed = time.time() - start
print(f'\n✅ Parsed {parsed:,} lines in {elapsed/60:.1f} minutes')
print(f'Unique clusters (= log keys): {len(miner.drain.clusters)}')
print(f'Lines without block_id: {no_block:,}')

# Sanity: verify log_keys in the CSV match cluster count
import subprocess
unique_keys = subprocess.check_output(
    "tail -n +2 /content/parsed_logs.csv | cut -d',' -f2 | sort -u | wc -l",
    shell=True
).decode().strip()
print(f'Unique log_keys in CSV: {unique_keys}')

Final parse: 100%|██████████| 11175629/11175629 [05:50<00:00, 31898.21it/s]



✅ Parsed 11,175,629 lines in 5.8 minutes
Unique clusters (= log keys): 34
Lines without block_id: 0
Unique log_keys in CSV: 34


In [27]:
# 4C.3 — Save templates
with open('/content/templates.csv', 'w', encoding='utf-8') as f:
    f.write('log_key,template\n')
    for cluster in miner.drain.clusters:
        t = cluster.get_template().replace('"', '""')
        f.write(f'{cluster.cluster_id},"{t}"\n')
print(f'✅ Templates saved: {len(miner.drain.clusters)}')

✅ Templates saved: 34


In [28]:
# 4C.4 — Inspect
clusters = sorted(miner.drain.clusters, key=lambda c: c.size, reverse=True)
print(f'{"ID":<6} {"Size":<10} {"Template"}')
print('-' * 110)
for c in clusters[:20]:
    print(f'{c.cluster_id:<6} {c.size:<10} {c.get_template()[:100]}')

ID     Size       Template
--------------------------------------------------------------------------------------------------------------
1      1723232    DATE NUM NUM INFO dfs.DataNode$DataXceiver: Receiving block BLK src: IP dest: IP
5      1719741    DATE NUM NUM INFO dfs.FSNamesystem: BLOCK* NameSystem.addStoredBlock: blockMap updated: IP:NUM is ad
3      1706728    DATE NUM NUM INFO dfs.DataNode$PacketResponder: PacketResponder NUM for block BLK <*>
4      1706514    DATE NUM NUM INFO dfs.DataNode$PacketResponder: Received block BLK of size NUM from IP
7      1409080    DATE NUM NUM INFO <*> <*> <*> <*> <*> <*>
24     1402854    DATE NUM NUM INFO dfs.FSNamesystem: BLOCK* <*> <*> <*> <*> <*> <*> <*> IP:50010
2      695097     DATE NUM NUM INFO <*> <*> <*> <*> BLK
9      428726     DATE NUM NUM INFO dfs.DataNode$DataXceiver: IP:NUM Served block BLK to IP
27     356207     DATE NUM NUM WARN dfs.DataNode$DataXceiver: IP:NUM:Got exception while serving BLK to IP:
6      7097       DAT

In [29]:
import numpy as np

clusters = sorted(miner.drain.clusters, key=lambda c: c.size, reverse=True)
sizes = [c.size for c in clusters]
total = sum(sizes)

print(f'Total clusters: {len(sizes)}')
print(f'Total lines:    {total:,}')
print(f'Max cluster:    {max(sizes):,}')
print(f'Min cluster:    {min(sizes):,}')
print()
print(f'Clusters with size == 1:      {sum(1 for s in sizes if s == 1)}')
print(f'Clusters with size <= 10:     {sum(1 for s in sizes if s <= 10)}')
print(f'Clusters with size <= 100:    {sum(1 for s in sizes if s <= 100)}')
print(f'Clusters with size <= 1000:   {sum(1 for s in sizes if s <= 1000)}')
print()

cumsum = np.cumsum(sizes)
for pct in [0.50, 0.90, 0.95, 0.99, 0.999]:
    n = int(np.searchsorted(cumsum, pct * total) + 1)
    print(f'{n:>3} clusters cover {pct*100:>5.1f}% of all lines')

Total clusters: 34
Total lines:    11,175,629
Max cluster:    1,723,232
Min cluster:    3

Clusters with size == 1:      0
Clusters with size <= 10:     8
Clusters with size <= 100:    17
Clusters with size <= 1000:   19

  4 clusters cover  50.0% of all lines
  7 clusters cover  90.0% of all lines
  8 clusters cover  95.0% of all lines
  9 clusters cover  99.0% of all lines
 12 clusters cover  99.9% of all lines


In [30]:
import shutil, time, os

pairs = [
    (LOCAL_PARSED, '/content/drive/MyDrive/module1/data/processed/parsed_logs.csv'),
    (LOCAL_TEMPLATES, '/content/drive/MyDrive/module1/data/processed/templates.csv'),
]
for src, dst in pairs:
    print(f'Copying {os.path.basename(src)}...')
    start = time.time()
    shutil.copy2(src, dst)
    print(f'  ✅ {os.path.getsize(dst)/(1024**2):.1f} MB in {time.time()-start:.1f}s')

Copying parsed_logs.csv...
  ✅ 368.2 MB in 5.3s
Copying templates.csv...
  ✅ 0.0 MB in 0.4s


In [31]:
import os
proc = '/content/drive/MyDrive/module1/data/processed/'
print('Contents of module1/data/processed/:')
for f in sorted(os.listdir(proc)):
    p = os.path.join(proc, f)
    print(f'  {f}: {os.path.getsize(p)/(1024**2):.1f} MB')

Contents of module1/data/processed/:
  parsed_logs.csv: 368.2 MB
  templates.csv: 0.0 MB


In [32]:
import pandas as pd

labels_df = pd.read_csv('/content/drive/MyDrive/module1/data/raw/anomaly_label.csv')

print('Shape:', labels_df.shape)
print('Columns:', list(labels_df.columns))
print('\nFirst 5 rows:')
print(labels_df.head().to_string())
print('\nLabel value counts:')
print(labels_df.iloc[:, 1].value_counts())
print(f'\nAnomaly rate: {100 * (labels_df.iloc[:, 1] == "Anomaly").mean():.2f}%')

Shape: (575061, 2)
Columns: ['BlockId', 'Label']

First 5 rows:
                    BlockId    Label
0  blk_-1608999687919862906   Normal
1   blk_7503483334202473044   Normal
2  blk_-3544583377289625738  Anomaly
3  blk_-9073992586687739851   Normal
4   blk_7854771516489510256   Normal

Label value counts:
Label
Normal     558223
Anomaly     16838
Name: count, dtype: int64

Anomaly rate: 2.93%


In [33]:
import os
from collections import defaultdict
from tqdm import tqdm
import numpy as np

PARSED_CSV = '/content/parsed_logs.csv'

# If local copy is gone (session restarted), pull from Drive
if not os.path.exists(PARSED_CSV):
    print('Local copy missing — copying from Drive (may take a minute)...')
    import shutil
    shutil.copy2(
        '/content/drive/MyDrive/module1/data/processed/parsed_logs.csv',
        PARSED_CSV
    )
    print('✅ Local copy restored.')

# Count total rows for progress bar
print('Counting rows for progress bar...')
with open(PARSED_CSV, 'rb') as f:
    total_lines = sum(1 for _ in f)
print(f'Total rows: {total_lines:,}')

# Build block_id -> [log_keys]
block_sequences = defaultdict(list)
line_count = 0
skipped = 0

with open(PARSED_CSV, 'r') as f:
    header = f.readline()   # skip header
    for line in tqdm(f, total=total_lines - 1, desc='Grouping by block'):
        parts = line.rstrip('\n').split(',')
        if len(parts) != 3:
            skipped += 1
            continue
        _line_no, log_key_str, block_id = parts
        if not block_id:
            skipped += 1
            continue
        block_sequences[block_id].append(int(log_key_str))
        line_count += 1

print(f'\n✅ Done')
print(f'Unique blocks:          {len(block_sequences):,}')
print(f'Total log keys read:    {line_count:,}')
print(f'Skipped rows:           {skipped:,}')

# Sequence length stats
lengths = [len(v) for v in block_sequences.values()]
print(f'\nSequence length stats:')
print(f'  Min:    {min(lengths)}')
print(f'  Max:    {max(lengths)}')
print(f'  Mean:   {np.mean(lengths):.1f}')
print(f'  Median: {int(np.median(lengths))}')
print(f'  P95:    {int(np.percentile(lengths, 95))}')
print(f'  P99:    {int(np.percentile(lengths, 99))}')

Counting rows for progress bar...
Total rows: 11,175,630


Grouping by block: 100%|██████████| 11175629/11175629 [00:22<00:00, 500930.25it/s]



✅ Done
Unique blocks:          575,061
Total log keys read:    11,175,629
Skipped rows:           0

Sequence length stats:
  Min:    2
  Max:    298
  Mean:   19.4
  Median: 19
  P95:    28
  P99:    33


In [34]:
label_dict = dict(zip(labels_df['BlockId'], labels_df['Label']))

blocks_in_parsed = set(block_sequences.keys())
blocks_in_labels = set(label_dict.keys())

only_in_parsed = blocks_in_parsed - blocks_in_labels
only_in_labels = blocks_in_labels - blocks_in_parsed
common         = blocks_in_parsed & blocks_in_labels

print(f'Blocks in parsed data:  {len(blocks_in_parsed):,}')
print(f'Blocks in labels file:  {len(blocks_in_labels):,}')
print(f'Common (in both):       {len(common):,}')
print(f'\nOnly in parsed (no label):   {len(only_in_parsed):,}')
print(f'Only in labels (no logs):    {len(only_in_labels):,}')

# Show a few examples of mismatches if any
if only_in_parsed:
    print('\nExamples of blocks in parsed but not labeled:')
    for b in list(only_in_parsed)[:5]:
        print(f'  {b}  (seq length: {len(block_sequences[b])})')

if only_in_labels:
    print('\nExamples of blocks in labels but not in parsed:')
    for b in list(only_in_labels)[:5]:
        print(f'  {b}')

Blocks in parsed data:  575,061
Blocks in labels file:  575,061
Common (in both):       575,061

Only in parsed (no label):   0
Only in labels (no logs):    0


In [35]:
import os
from tqdm import tqdm

OUT_DIR = '/content/sequences'
os.makedirs(OUT_DIR, exist_ok=True)

seq_path = f'{OUT_DIR}/deeplog_input.txt'
label_path = f'{OUT_DIR}/deeplog_labels.txt'

written = 0
label_counts = {'Normal': 0, 'Anomaly': 0, 'Unknown': 0}

with open(seq_path, 'w') as f_seq, open(label_path, 'w') as f_lab:
    f_lab.write('block_id,label\n')

    for block_id, keys in tqdm(block_sequences.items(), desc='Writing'):
        label = label_dict.get(block_id, 'Unknown')
        label_counts[label] += 1

        if label == 'Unknown':
            continue  # skip unlabeled

        f_seq.write(' '.join(map(str, keys)) + '\n')
        f_lab.write(f'{block_id},{label}\n')
        written += 1

print(f'\n✅ Sequences written: {written:,}')
print(f'Label distribution:  {label_counts}')

# File sizes
for p in [seq_path, label_path]:
    size_mb = os.path.getsize(p) / (1024**2)
    print(f'  {os.path.basename(p)}: {size_mb:.1f} MB')

Writing: 100%|██████████| 575061/575061 [00:03<00:00, 178921.69it/s]


✅ Sequences written: 575,061
Label distribution:  {'Normal': 558223, 'Anomaly': 16838, 'Unknown': 0}
  deeplog_input.txt: 23.0 MB
  deeplog_labels.txt: 17.2 MB


In [36]:
max_key = max(max(keys) for keys in block_sequences.values() if keys)

with open(f'{OUT_DIR}/vocab_info.txt', 'w') as f:
    f.write(f'vocab_size={max_key}\n')
    f.write(f'num_templates={len(miner.drain.clusters)}\n')
    f.write(f'num_sequences={written}\n')
    f.write(f'num_blocks_total={len(block_sequences)}\n')
    f.write(f'mean_sequence_length={sum(len(v) for v in block_sequences.values())/len(block_sequences):.2f}\n')
    f.write(f'max_sequence_length={max(len(v) for v in block_sequences.values())}\n')

print('vocab_info.txt contents:')
print('-' * 40)
with open(f'{OUT_DIR}/vocab_info.txt') as f:
    print(f.read())

vocab_info.txt contents:
----------------------------------------
vocab_size=34
num_templates=34
num_sequences=575061
num_blocks_total=575061
mean_sequence_length=19.43
max_sequence_length=298



In [37]:
import random

# Build reverse map: cluster_id -> template text
template_rev = {c.cluster_id: c.get_template() for c in miner.drain.clusters}

# Load a small sample of sequences + labels
with open(seq_path) as f:
    all_seqs = [line.strip() for line in f]

with open(label_path) as f:
    f.readline()  # skip header
    all_labels = [line.strip() for line in f]

print(f'Loaded {len(all_seqs):,} sequences, {len(all_labels):,} labels\n')

# Pick 2 random Normal + 2 random Anomaly to inspect
normal_idx = [i for i, l in enumerate(all_labels) if l.endswith('Normal')]
anomaly_idx = [i for i, l in enumerate(all_labels) if l.endswith('Anomaly')]

random.seed(42)
picks = random.sample(normal_idx, 2) + random.sample(anomaly_idx, 2)

for idx in picks:
    block_id, label = all_labels[idx].split(',')
    keys = [int(x) for x in all_seqs[idx].split()]
    print(f'=== {block_id}  [{label}]  ({len(keys)} events) ===')
    for k in keys[:12]:
        tmpl = template_rev.get(k, 'UNKNOWN')
        print(f'  {k:>3}  {tmpl[:95]}')
    if len(keys) > 12:
        print(f'  ... ({len(keys) - 12} more events)')
    print()

Loaded 575,061 sequences, 575,061 labels

=== blk_-5768125218786247961  [Normal]  (23 events) ===
    2  DATE NUM NUM INFO <*> <*> <*> <*> BLK
    1  DATE NUM NUM INFO dfs.DataNode$DataXceiver: Receiving block BLK src: IP dest: IP
    1  DATE NUM NUM INFO dfs.DataNode$DataXceiver: Receiving block BLK src: IP dest: IP
    1  DATE NUM NUM INFO dfs.DataNode$DataXceiver: Receiving block BLK src: IP dest: IP
    5  DATE NUM NUM INFO dfs.FSNamesystem: BLOCK* NameSystem.addStoredBlock: blockMap updated: IP:NUM 
    5  DATE NUM NUM INFO dfs.FSNamesystem: BLOCK* NameSystem.addStoredBlock: blockMap updated: IP:NUM 
    5  DATE NUM NUM INFO dfs.FSNamesystem: BLOCK* NameSystem.addStoredBlock: blockMap updated: IP:NUM 
    3  DATE NUM NUM INFO dfs.DataNode$PacketResponder: PacketResponder NUM for block BLK <*>
    4  DATE NUM NUM INFO dfs.DataNode$PacketResponder: Received block BLK of size NUM from IP
    3  DATE NUM NUM INFO dfs.DataNode$PacketResponder: PacketResponder NUM for block BLK <*>
    

In [38]:
import shutil, os, time

drive_dir = '/content/drive/MyDrive/module1/data/processed/'
files_to_copy = ['deeplog_input.txt', 'deeplog_labels.txt', 'vocab_info.txt']

for fname in files_to_copy:
    src = f'{OUT_DIR}/{fname}'
    dst = f'{drive_dir}{fname}'
    print(f'Copying {fname}...')
    start = time.time()
    shutil.copy2(src, dst)
    size_mb = os.path.getsize(dst) / (1024**2)
    print(f'  ✅ {size_mb:.1f} MB in {time.time()-start:.1f}s')

print('\n═══════════════════════════════════════════════════')
print('FINAL CONTENTS of module1/data/processed/:')
print('═══════════════════════════════════════════════════')
for f in sorted(os.listdir(drive_dir)):
    p = os.path.join(drive_dir, f)
    size_mb = os.path.getsize(p) / (1024**2)
    print(f'  {f:<30} {size_mb:>8.2f} MB')

Copying deeplog_input.txt...
  ✅ 23.0 MB in 0.3s
Copying deeplog_labels.txt...
  ✅ 17.2 MB in 0.4s
Copying vocab_info.txt...
  ✅ 0.0 MB in 0.0s

═══════════════════════════════════════════════════
FINAL CONTENTS of module1/data/processed/:
═══════════════════════════════════════════════════
  deeplog_input.txt                 23.01 MB
  deeplog_labels.txt                17.23 MB
  parsed_logs.csv                  368.18 MB
  templates.csv                      0.00 MB
  vocab_info.txt                     0.00 MB


In [39]:
import os
import numpy as np

DRIVE_DIR = '/content/drive/MyDrive/module1/data/processed/'

print('=' * 65)
print('MODULE 1 — FINAL HANDOFF VERIFICATION')
print('=' * 65)

# --- 1. All required files exist ---
required = [
    'deeplog_input.txt',
    'deeplog_labels.txt',
    'templates.csv',
    'vocab_info.txt',
    'parsed_logs.csv',
    'README.md',
]
print('\n[1] Required files present:')
missing = []
for f in required:
    p = os.path.join(DRIVE_DIR, f)
    if os.path.exists(p):
        size_mb = os.path.getsize(p) / (1024**2)
        print(f'    ✅ {f:<25} {size_mb:>8.2f} MB')
    else:
        print(f'    ❌ {f:<25} MISSING')
        missing.append(f)
if missing:
    raise SystemExit(f'\n❌ Missing files: {missing}. Stop here.')

# --- 2. Load sequences and labels (simulating Member 2) ---
print('\n[2] Loading sequences and labels...')

with open(os.path.join(DRIVE_DIR, 'deeplog_input.txt')) as f:
    sequences = [line.strip() for line in f]

with open(os.path.join(DRIVE_DIR, 'deeplog_labels.txt')) as f:
    f.readline()  # header
    labels = [line.strip() for line in f]

print(f'    Sequences loaded: {len(sequences):,}')
print(f'    Labels loaded:    {len(labels):,}')

assert len(sequences) == len(labels), 'MISMATCH between sequences and labels!'
print(f'    ✅ Line counts match: {len(sequences):,}')

# --- 3. Parse and validate log keys ---
print('\n[3] Validating log keys...')
parsed_seqs = [[int(k) for k in s.split()] for s in sequences]
all_keys = set(k for seq in parsed_seqs for k in seq)

print(f'    Unique log keys: {len(all_keys)}')
print(f'    Min key: {min(all_keys)}')
print(f'    Max key: {max(all_keys)}')
print(f'    Range valid (1–34): {min(all_keys) == 1 and max(all_keys) == 34}')
print(f'    No key == 0 (padding safe): {0 not in all_keys}')

# --- 4. Sequence length statistics ---
print('\n[4] Sequence length statistics:')
lengths = [len(s) for s in parsed_seqs]
print(f'    Min:    {min(lengths)}')
print(f'    Max:    {max(lengths)}')
print(f'    Mean:   {np.mean(lengths):.2f}')
print(f'    Median: {int(np.median(lengths))}')

# --- 5. Label distribution ---
print('\n[5] Label distribution:')
n_normal = sum(1 for l in labels if l.endswith('Normal'))
n_anomaly = sum(1 for l in labels if l.endswith('Anomaly'))
total = n_normal + n_anomaly
print(f'    Normal:  {n_normal:,}  ({100*n_normal/total:.2f}%)')
print(f'    Anomaly: {n_anomaly:,}  ({100*n_anomaly/total:.2f}%)')
print(f'    Total:   {total:,}')

# --- 6. Vocabulary consistency ---
print('\n[6] Vocabulary info file:')
with open(os.path.join(DRIVE_DIR, 'vocab_info.txt')) as f:
    vocab_txt = f.read()
print(vocab_txt)

# --- 7. Template lookup sanity ---
print('\n[7] Template lookup sanity:')
import csv
with open(os.path.join(DRIVE_DIR, 'templates.csv')) as f:
    reader = csv.DictReader(f)
    templates = {int(row['log_key']): row['template'] for row in reader}
print(f'    Templates loaded: {len(templates)}')
print(f'    Sample — key 1: {templates.get(1, "N/A")[:80]}')
print(f'    Sample — key 4: {templates.get(4, "N/A")[:80]}')

# --- 8. Alignment spot-check ---
print('\n[8] Spot-check: sequence 0 ↔ label 0 alignment')
print(f'    Sequence[0] length: {len(parsed_seqs[0])}')
print(f'    Sequence[0] first 10 keys: {parsed_seqs[0][:10]}')
print(f'    Label[0]: {labels[0]}')

print('\n' + '=' * 65)
print('✅ VERIFICATION PASSED — MODULE 1 IS READY FOR HANDOFF')
print('=' * 65)
print('\nYou can now copy module1/ into the shared LogSentry_Capstone/ folder.')

MODULE 1 — FINAL HANDOFF VERIFICATION

[1] Required files present:
    ✅ deeplog_input.txt            23.01 MB
    ✅ deeplog_labels.txt           17.23 MB
    ✅ templates.csv                 0.00 MB
    ✅ vocab_info.txt                0.00 MB
    ✅ parsed_logs.csv             368.18 MB
    ✅ README.md                     0.01 MB

[2] Loading sequences and labels...
    Sequences loaded: 575,061
    Labels loaded:    575,061
    ✅ Line counts match: 575,061

[3] Validating log keys...
    Unique log keys: 34
    Min key: 1
    Max key: 34
    Range valid (1–34): True
    No key == 0 (padding safe): True

[4] Sequence length statistics:
    Min:    2
    Max:    298
    Mean:   19.43
    Median: 19

[5] Label distribution:
    Normal:  558,223  (97.07%)
    Anomaly: 16,838  (2.93%)
    Total:   575,061

[6] Vocabulary info file:
vocab_size=34
num_templates=34
num_sequences=575061
num_blocks_total=575061
mean_sequence_length=19.43
max_sequence_length=298


[7] Template lookup sanity:
    

In [40]:
import os

shared_root = '/content/drive/MyDrive/LogSentry_Capstone'

print('Shared folder accessible?', os.path.exists(shared_root))
if os.path.exists(shared_root):
    print('\nContents:')
    for item in sorted(os.listdir(shared_root)):
        print(f'  {item}')

Shared folder accessible? True

Contents:
  01_Dataset
  02_Parsed_Data
  03_Models
  04_Results


In [41]:
import os
import shutil

# Source: where Colab saved it by default
src = '/content/drive/MyDrive/Colab Notebooks/member1_parsing.ipynb'

# Destination: inside your module1 folder
dst = '/content/drive/MyDrive/module1/code/member1_parsing.ipynb'

# Check the notebook exists at src
if os.path.exists(src):
    shutil.copy2(src, dst)
    print(f'✅ Notebook copied:')
    print(f'   From: {src}')
    print(f'   To:   {dst}')
    print(f'   Size: {os.path.getsize(dst)/(1024**2):.2f} MB')
else:
    print(f'❌ Notebook not found at: {src}')
    print('\nListing /content/drive/MyDrive/Colab Notebooks/ to find it:')
    notebook_dir = '/content/drive/MyDrive/Colab Notebooks'
    if os.path.exists(notebook_dir):
        for f in os.listdir(notebook_dir):
            if f.endswith('.ipynb'):
                print(f'   {f}')
    else:
        print('   Folder does not exist.')

❌ Notebook not found at: /content/drive/MyDrive/Colab Notebooks/member1_parsing.ipynb

Listing /content/drive/MyDrive/Colab Notebooks/ to find it:
   Folder does not exist.


In [42]:
import os

print('Searching your Drive for *.ipynb files...\n')
found = []
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for f in files:
        if f.endswith('.ipynb'):
            p = os.path.join(root, f)
            size_kb = os.path.getsize(p) / 1024
            found.append((p, size_kb))

if found:
    for p, size_kb in found:
        print(f'  {p}  ({size_kb:.1f} KB)')
else:
    print('  No .ipynb files found anywhere in your Drive.')
    print('  → The notebook has NOT been saved to Drive yet.')
    print('  → In Colab: File → Save (Ctrl+S), then re-run this cell.')

Searching your Drive for *.ipynb files...

  /content/drive/MyDrive/member1_parsing.ipynb  (69.1 KB)


In [43]:
import os
import shutil

src = '/content/drive/MyDrive/member1_parsing.ipynb'
dst = '/content/drive/MyDrive/module1/code/member1_parsing.ipynb'

if os.path.exists(src):
    shutil.copy2(src, dst)
    size_kb = os.path.getsize(dst) / 1024
    print(f'✅ Notebook copied into module1/code/')
    print(f'   Source:      {src}')
    print(f'   Destination: {dst}')
    print(f'   Size:        {size_kb:.1f} KB')
else:
    print(f'❌ Not found at {src}')

✅ Notebook copied into module1/code/
   Source:      /content/drive/MyDrive/member1_parsing.ipynb
   Destination: /content/drive/MyDrive/module1/code/member1_parsing.ipynb
   Size:        69.1 KB


In [44]:
import os

base = '/content/drive/MyDrive/module1'
print('Full module1/ structure:')
for root, dirs, files in os.walk(base):
    level = root.replace(base, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    for f in sorted(files):
        p = os.path.join(root, f)
        size_mb = os.path.getsize(p) / (1024**2)
        print(f'{indent}  {f}  ({size_mb:.2f} MB)')

Full module1/ structure:
module1/
  data/
    raw/
      HDFS_v1.zip  (178.00 MB)
      anomaly_label.csv  (17.77 MB)
    processed/
      README.md  (0.01 MB)
      deeplog_input.txt  (23.01 MB)
      deeplog_labels.txt  (17.23 MB)
      parsed_logs.csv  (368.18 MB)
      templates.csv  (0.00 MB)
      vocab_info.txt  (0.00 MB)
  code/
    member1_parsing.ipynb  (0.07 MB)


In [45]:
import os

shared = '/content/drive/MyDrive/LogSentry_Capstone'
print('Shared folder exists?', os.path.exists(shared))
if os.path.exists(shared):
    for item in sorted(os.listdir(shared)):
        print(f'  {item}')

Shared folder exists? True
  01_Dataset
  02_Parsed_Data
  03_Models
  04_Results


In [46]:
import shutil, os, time

SRC      = '/content/drive/MyDrive/module1'
DST_ROOT = '/content/drive/MyDrive/LogSentry_Capstone/02_Parsed_Data'
DST      = os.path.join(DST_ROOT, 'module1')

# Safety: verify source and destination root both exist
assert os.path.exists(SRC),      f'Source missing: {SRC}'
assert os.path.exists(DST_ROOT), f'Destination parent missing: {DST_ROOT}'

# If destination already exists, rename to avoid overwriting
if os.path.exists(DST):
    print(f'⚠️  {DST} already exists. Renaming this copy with timestamp.')
    DST = DST + '_' + time.strftime('%Y%m%d_%H%M%S')

print(f'Copying:')
print(f'  from {SRC}')
print(f'  to   {DST}\n')

start = time.time()
copied_files = 0
copied_bytes = 0

for root, dirs, files in os.walk(SRC):
    for f in files:
        src_path = os.path.join(root, f)
        rel = os.path.relpath(src_path, SRC)
        dst_path = os.path.join(DST, rel)

        os.makedirs(os.path.dirname(dst_path), exist_ok=True)
        shutil.copy2(src_path, dst_path)

        size_mb = os.path.getsize(dst_path) / (1024**2)
        copied_files += 1
        copied_bytes += os.path.getsize(dst_path)
        print(f'  ✅ {rel:<45} {size_mb:>8.2f} MB')

elapsed = time.time() - start
total_mb = copied_bytes / (1024**2)
print(f'\n✅ Copied {copied_files} files ({total_mb:.1f} MB) in {elapsed:.1f}s')
print(f'   Destination: {DST}')

Copying:
  from /content/drive/MyDrive/module1
  to   /content/drive/MyDrive/LogSentry_Capstone/02_Parsed_Data/module1

  ✅ data/raw/HDFS_v1.zip                            178.00 MB
  ✅ data/raw/anomaly_label.csv                       17.77 MB
  ✅ data/processed/parsed_logs.csv                  368.18 MB
  ✅ data/processed/templates.csv                      0.00 MB
  ✅ data/processed/deeplog_input.txt                 23.01 MB
  ✅ data/processed/deeplog_labels.txt                17.23 MB
  ✅ data/processed/vocab_info.txt                     0.00 MB
  ✅ data/processed/README.md                          0.01 MB
  ✅ code/member1_parsing.ipynb                        0.07 MB

✅ Copied 9 files (604.3 MB) in 13.2s
   Destination: /content/drive/MyDrive/LogSentry_Capstone/02_Parsed_Data/module1


In [47]:
import os

DST = '/content/drive/MyDrive/LogSentry_Capstone/02_Parsed_Data/module1'

print(f'Verifying contents of:\n  {DST}\n')
print('=' * 65)

total_mb = 0
for root, dirs, files in os.walk(DST):
    level = root.replace(DST, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    for f in sorted(files):
        p = os.path.join(root, f)
        size_mb = os.path.getsize(p) / (1024**2)
        total_mb += size_mb
        print(f'{indent}  {f}  ({size_mb:.2f} MB)')

print('=' * 65)
print(f'Total: {total_mb:.1f} MB')

Verifying contents of:
  /content/drive/MyDrive/LogSentry_Capstone/02_Parsed_Data/module1

module1/
  data/
    raw/
      HDFS_v1.zip  (178.00 MB)
      anomaly_label.csv  (17.77 MB)
    processed/
      README.md  (0.01 MB)
      deeplog_input.txt  (23.01 MB)
      deeplog_labels.txt  (17.23 MB)
      parsed_logs.csv  (368.18 MB)
      templates.csv  (0.00 MB)
      vocab_info.txt  (0.00 MB)
  code/
    member1_parsing.ipynb  (0.07 MB)
Total: 604.3 MB
